In [1]:
import requests, json

def ask(prompt, system=None, temperature=0.7, model="llama3.2:3b"):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    })
    return r.json()["message"]["content"]

In [2]:
print("ZERO-SHOT:\n", ask("Classify the sentiment of: 'The battery life is terrible.'"))

few_shot = """Classify the sentiment as Positive or Negative.
Review: "I love this phone." -> Positive
Review: "Worst purchase ever." -> Negative
Review: "Works great, fast shipping." -> Positive
Review: "The battery life is terrible." ->"""
print("\nFEW-SHOT:\n", ask(few_shot))

ZERO-SHOT:
 The sentiment of the statement "The battery life is terrible" is negative.

FEW-SHOT:
 Negative

The review mentions that the battery life is "terrible", which indicates a negative sentiment.


Both said Negative. With examples, it answered in the same short format, but still added an explanation.

In [3]:
q = "I have 3 apples. I eat one, then buy twice as many as I have left, then give away 4. How many do I have?"
print("WITHOUT:\n", ask(q, temperature=0))
print("\nWITH:\n", ask(q + " Let's think step by step.", temperature=0))

WITHOUT:
 Let's break it down step by step:

1. You start with 3 apples.
2. You eat one, so you have 3 - 1 = 2 apples left.
3. You buy twice as many as you have left, which is 2 x 2 = 4 apples. Now you have 2 + 4 = 6 apples.
4. You give away 4 apples, so you have 6 - 4 = 2 apples left.

You now have 2 apples.

WITH:
 Let's break it down step by step:

1. You start with 3 apples.
2. You eat one, so you have 3 - 1 = 2 apples left.
3. You buy twice as many as you have left, which is 2 x 2 = 4 apples. Now you have 2 + 4 = 6 apples.
4. You give away 4 apples, so you have 6 - 4 = 2 apples left.

You now have 2 apples.


Both got the right answer (2). The model already thinks step by step on its own, so the extra phrase didn't change anything.

In [4]:
code = "Review this code: def avg(x): return sum(x)/len(x)"
print("NO ROLE:\n", ask(code, temperature=0))
print("\nSENIOR DEV:\n", ask(code, system="You are a senior Python developer reviewing code.", temperature=0))

NO ROLE:
 **Code Review**

The provided code defines a simple function `avg` that calculates the average of a list of numbers `x`. Here's a review of the code:

**Positive aspects:**

1. **Concise**: The code is very concise and directly calculates the average of the input list.
2. **Easy to understand**: The code is straightforward and easy to comprehend, even for those without prior programming experience.

**Suggestions for improvement:**

1. **Error handling**: The function does not handle cases where the input list is empty. If the list is empty, the function will raise a `ZeroDivisionError`. Consider adding a check to handle this case.
2. **Type hinting**: Adding type hints for the function parameters and return type can improve code readability and make it easier for others to understand the function's behavior.
3. **Docstring**: Adding a docstring to the function can provide a brief description of its purpose and usage.
4. **Input validation**: Consider adding input validation 

Both caught the empty-list crash. The senior dev role mostly changed the tone, not what it found.

In [5]:
out = ask("Reply only with a JSON object with keys 'sentiment' and 'confidence'. "
          "Text: 'The battery life is terrible.'", temperature=0)
print(out)
try:
    print("\nValid JSON:", json.loads(out))
except json.JSONDecodeError:
    print("\nNOT valid JSON: the model added extra text around it")

{"sentiment": "negative", "confidence": 0.9}

Valid JSON: {'sentiment': 'negative', 'confidence': 0.9}


It returned clean JSON that Python could read. The confidence number is made up by the model, not a real measurement.

In [6]:
p = "Write a one-line slogan for a coffee shop."
print("TEMPERATURE 0:")
for _ in range(3): print(" -", ask(p, temperature=0))
print("\nTEMPERATURE 1.5:")
for _ in range(3): print(" -", ask(p, temperature=1.5))

TEMPERATURE 0:
 - "Wake up to the perfect blend."
 - "Wake up to the perfect blend."
 - "Wake up to the perfect blend."

TEMPERATURE 1.5:
 - "Wake up to a world of flavor, one cup at a time."
 - "Brewing up connections, one cup at a time."
 - "Wake up to the art of good taste, one cup at a time."


At 0, all three slogans were the same. At 1.5, each one was different.